In [ ]:
from MobilityHubDataObjects import *
import geopandas as gpd
import folium
import datetime as dt

In [ ]:
CONFIG_MAP_AREA_PATH = "map_area_path"
CONFIG_MAP_AREA_PROJECTED_CRS = "map_area_projected_crs"
CONFIG_GTFS_TRANSITLAND_URL = "gtfs_url"
CONFIG_GTFS_TRANSITLAND_KEY_PATH = "gtfs_key_path"
CONFIG_GTFS_CACHE_FOLDER = "gtfs_cache_folder"
CONFIG_GTFS_OVERRIDE_SHEET = "gtfs_override_sheet"
CONFIG_FTA_FACILITY_INVENTORY_PATH = "fta_facility_inventory_path"
CONFIG_TIGER_STATES_PATH = "tiger_states_path"
CONFIG_CITYBIKES_URL = "citybikes_url"
CONFIG_AFDC_API_KEY = "afdc_api_key"
CONFIG_AFDC_URL = "afdc_url"
CONFIG_OSM_CACHE_FOLDER = "osm_cache_folder"
CONFIG_EPA_EJSCREEN_PATH = "epa_ejscreen_path"
CONFIG_SMART_LOCATION_PATH = "smart_location_path"

GEODESIC_CRS = 4326

WAYNE_COUNTY_PATH = "./rawData/waynecounty.geojson"
WAYNE_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/wayne_county"
WAYNE_COUNTY_CRS = 6498

COOK_COUNTY_PATH = "./rawData/CookCounty.geojson"
COOK_COUNTY_GTFS_CACHE = "./cache/gtfs_cache/chicagoland"
COOK_COUNTY_GTFS_OVERRIDE = "./cache/gtfs_cache/chicagoland/override_feeds.csv"
COOK_COUNTY_CRS = 26971

# COMPLETE CONFIG HERE
CONFIG = {
    CONFIG_MAP_AREA_PATH: COOK_COUNTY_PATH,
    CONFIG_MAP_AREA_PROJECTED_CRS: COOK_COUNTY_CRS, # This should be an epsg number with the units in meters
    CONFIG_GTFS_TRANSITLAND_URL: "https://transit.land/api/v2/rest/feeds.json",
    CONFIG_GTFS_TRANSITLAND_KEY_PATH: "./rawData/TRANSITLAND_KEY",
    CONFIG_GTFS_CACHE_FOLDER: COOK_COUNTY_GTFS_CACHE,
    CONFIG_GTFS_OVERRIDE_SHEET: COOK_COUNTY_GTFS_OVERRIDE,
    CONFIG_FTA_FACILITY_INVENTORY_PATH: "./rawData/2022 Facility Inventory.xlsx",
    CONFIG_TIGER_STATES_PATH: "./rawData/tl_2023_us_state/tl_2023_us_state.shp",
    CONFIG_CITYBIKES_URL: "http://api.citybik.es/",
    CONFIG_OSM_CACHE_FOLDER: "./cache/osmnx_cache",
    CONFIG_AFDC_URL: "https://developer.nrel.gov/api/alt-fuel-stations/v1/nearest.geojson",
    CONFIG_AFDC_API_KEY: "./rawData/AFDC_API_KEY",
    CONFIG_EPA_EJSCREEN_PATH: "rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb",
    CONFIG_SMART_LOCATION_PATH: "rawData/SmartLocationDatabaseV3/SmartLocationDatabase.gdb",
}

In [ ]:
# Define map area
map_area = gpd.read_file(CONFIG[CONFIG_MAP_AREA_PATH]).to_crs(GEODESIC_CRS).loc[0,"geometry"]
map_area

In [ ]:
# Instantiate data objects
gtfs_instance = GTFSDataObject(
    CONFIG[CONFIG_GTFS_CACHE_FOLDER],
    CONFIG[CONFIG_GTFS_TRANSITLAND_URL],
    dt.timedelta(days=90),
    dt.time(hour=10), #TODO: need to handle tz
    dt.time(hour=15),
    min_headway=9000,
    api_key_path=CONFIG[CONFIG_GTFS_TRANSITLAND_KEY_PATH],
    gtfs_override_feeds_path=CONFIG[CONFIG_GTFS_OVERRIDE_SHEET]
)
fta_instance = FTAFacilityInventoryDataObject(
    CONFIG[CONFIG_FTA_FACILITY_INVENTORY_PATH],
    CONFIG[CONFIG_TIGER_STATES_PATH],
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)
citybikes_instance = CityBikesDataObject(CONFIG[CONFIG_CITYBIKES_URL])
afdc_instance = AFDCApiDataObject(CONFIG[CONFIG_AFDC_URL], CONFIG[CONFIG_AFDC_API_KEY],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
osm_bike_parking_instance = OSMDataObject(CONFIG[CONFIG_OSM_CACHE_FOLDER], {"amenity": ["bicycle_parking"]})
base_layer_instance = BaseLayer(
    [
        CensusModeshare(),
        CensusCarlessness(),
        CensusPopulation(),
        BaseLayerSmartLocation(CONFIG[CONFIG_SMART_LOCATION_PATH],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]),
        BaseLayerEjscreen(CONFIG[CONFIG_EPA_EJSCREEN_PATH]),
    ],
    "rawData/tl_2023_us_county/tl_2023_us_county.shp",
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    ColorMaps.EJSCREEN_ONLY_COLOR_MAP_ID
)
all_objects = {
    "GTFS": gtfs_instance,
    "FTA": fta_instance,
    "CITYBIKES": citybikes_instance,
    "AFDC": afdc_instance,
    "OSM BIKE PARKING": osm_bike_parking_instance,
    "BASE": base_layer_instance,
}
objects_load_order = (
    "BASE",
    "GTFS",
    "FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "CITYBIKES"
)
objects_must_await = (
    "GTFS"
)

In [ ]:
# Load data objects
for name, object_name in all_objects.items():
    print(name)
    if name in objects_must_await:
        await object_name.load_data(map_area, GEODESIC_CRS)
    else:
        object_name.load_data(map_area, GEODESIC_CRS)

In [ ]:
# Main Map
display_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
    prefer_canvas=True
)
for object_name in objects_load_order:
    print(object_name)
    all_objects[object_name].get_folium_plot().add_to(display_map)
#all_objects["GTFS"].get_folium_plot().add_to(display_map)
display_map

In [ ]:
base_layer_instance.gdf.to_file("test_base_layer_chicagoland.geojson")

In [ ]:
# Score Example
s = ScoreWrapper([gtfs_instance, citybikes_instance, osm_bike_parking_instance, afdc_instance], ["GTFS", "Citybikes", "OSM", "AFDC"], 1000, 6498)
gdf_tracts = ejscreen_instance.gdf
gdf_tracts["score"] = s.get_score_at_point_geoseries(gdf_tracts.centroid)

from MobilityHubDataObjects.utils import basic_circle_marker, get_scores_for_all_objects
import branca.colormap as cm

scores = get_scores_for_all_objects([gtfs_instance, citybikes_instance, osm_bike_parking_instance, afdc_instance], ["GTFS", "Citybikes", "OSM", "AFDC"])

linear = cm.LinearColormap(["green", "blue"], vmin=0, vmax=50)
score_map = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="Cartodb Positron",
    zoom_start=10,
)

folium.GeoJson(
    gdf_tracts, 
    style_function=lambda feature: {
        "fillColor": linear(feature["properties"]["score"])
    },
    popup=folium.GeoJsonPopup(fields=["score"])
).add_to(score_map)
folium.GeoJson(
    scores, 
    marker=basic_circle_marker("green"),
    style_function=lambda feature: {"fillColor": "orange" if feature["properties"]["type"] == "GTFS" else "green"},
    popup=folium.GeoJsonPopup(fields=["score", "type"])
).add_to(score_map)

score_map

In [ ]:
import osmnx as ox

In [ ]:
ox.features.features_from_point((34.1019028, -118.2984934), tags={"cycleway": True, "highway": "cycleway", "cycleway:left": True, "cycleway:right": True}, dist=20000)

In [ ]:
# Define the coordinates of the point of interest
lat, lon =  52.3718378, 4.8961713

# Define the search distance (500 meters)
distance = 500

# Retrieve the cycleway features within the specified distance
cycleways = ox.features.features_from_point((lat, lon), dist=distance, tags={'highway': 'cycleway'})


In [ ]:
cycleways